# Project 03: MNIST GAN Baseline (Bridge-Assisted)

Build a GAN in PyTorch and inject a NeuroGebra custom activation using `to_pytorch`.


## Dataset Source and Download Instructions

- Dataset: MNIST handwritten digits
- Official source: http://yann.lecun.com/exdb/mnist/
- License: dataset terms from original creators
- Auto-download in this notebook: `torchvision.datasets.MNIST(download=True)`
- Manual fallback:
  1. Download files from the official source.
  2. Put them in `./data/MNIST/raw/`.
  3. Re-run data loading.


In [ ]:
%pip -q install neurogebra torch torchvision matplotlib


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from neurogebra import Expression
from neurogebra.bridges.pytorch_bridge import to_pytorch
from neurogebra.logging.dashboard import DashboardExporter
from neurogebra.logging.logger import LogLevel, TrainingLogger

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
ds = datasets.MNIST(root=Path("./data"), train=True, download=True, transform=transform)
ds = Subset(ds, np.arange(10000))
loader = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)


In [ ]:
mish_expr = Expression("mish", "x * tanh(log(1 + exp(x)))")
mish_layer = to_pytorch(mish_expr)

class Generator(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            mish_layer,
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCELoss()
opt_g = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

logger = TrainingLogger(level=LogLevel.DETAILED)
dashboard = DashboardExporter(path="./logs_gan_baseline/dashboard.html")
logger.add_backend(dashboard)

epochs = 4
latent_dim = 64
logger.on_train_start(total_epochs=epochs, total_samples=len(loader.dataset), batch_size=loader.batch_size)

g_hist = []
d_hist = []

for epoch in range(epochs):
    logger.on_epoch_start(epoch)
    g_loss_epoch = 0.0
    d_loss_epoch = 0.0

    for batch_idx, (real_imgs, _) in enumerate(loader):
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        real_targets = torch.ones(bs, 1, device=device)
        fake_targets = torch.zeros(bs, 1, device=device)

        # Train discriminator
        opt_d.zero_grad()
        real_out = D(real_imgs)
        d_real_loss = criterion(real_out, real_targets)

        z = torch.randn(bs, latent_dim, device=device)
        fake_imgs = G(z)
        fake_out = D(fake_imgs.detach())
        d_fake_loss = criterion(fake_out, fake_targets)

        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        opt_d.step()

        # Train generator
        opt_g.zero_grad()
        z = torch.randn(bs, latent_dim, device=device)
        gen_imgs = G(z)
        gen_out = D(gen_imgs)
        g_loss = criterion(gen_out, real_targets)
        g_loss.backward()
        opt_g.step()

        d_loss_epoch += float(d_loss.item())
        g_loss_epoch += float(g_loss.item())

        logger.on_batch_end(
            batch_idx,
            epoch=epoch,
            loss=float(g_loss.item()),
            metrics={"d_loss": float(d_loss.item())},
        )

    avg_g = g_loss_epoch / len(loader)
    avg_d = d_loss_epoch / len(loader)
    g_hist.append(avg_g)
    d_hist.append(avg_d)

    logger.on_epoch_end(
        epoch,
        metrics={"loss": avg_g, "val_loss": avg_d, "accuracy": 0.0, "val_accuracy": 0.0},
    )

logger.on_train_end(final_metrics={"g_loss": g_hist[-1], "d_loss": d_hist[-1]})
dashboard_path = dashboard.save()
print("Dashboard:", dashboard_path)


In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(16, 64, device=device)
    samples = G(z).cpu().numpy()

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i, 0], cmap="gray")
    ax.axis("off")
plt.suptitle("GAN Samples")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(g_hist, label="generator_loss")
plt.plot(d_hist, label="discriminator_loss")
plt.legend()
plt.title("GAN Training Curves")
plt.show()
